<a href="https://colab.research.google.com/github/JCF-the-creator/Multi_Motorsport_Analytics_Desempenho_e_Estrategia/blob/main/Codigo_PY_Motogp/motogp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 540.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 102.6 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [ ]:
import os
import requests
from pathlib import Path
from re
from genericpath import exists
import pdfplumber
import pandas as pd
import zipfile
import json

HEADERS = {"User-Agent": "Mozilla/5.0"}

#anos
YEARS1 = [2002, 2003, 2004, 2005, 2006, 2007, 2008]

#MUDA A POSIÇÃO DO LINK
YEARS2 = [2009, 2010,
        2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019,
        2020, 2021, 2022, 2023, 2024, 2025, 2026]

# Lista de circuitos (slugs oficiais usados nos PDFs)
CIRCUITS = ["VAL","CAT","BRNO","MIS","SEP","QAT","AUS","ARG","JPN","ITA","GER","NED","GBR","USA","POR","ARA","SMR","MAL","FRA"]

#etapas
ETAPAS = ["RAC","QP","Q1","Q2","QP1","QP2"]

#muda com alguns anos
DOC = ["classification","Classification"]

for y in YEARS1:
  for c in CIRCUITS:
    for e in ETAPAS:
      for d in DOC:

          url = f"https://resources.motogp.com/files/results/{y}/MotoGP/{c}/{e}/{d}.pdf"

          pasta_destino = Path(f"motogp/{str(y)}/{c}")
          pasta_destino.mkdir(parents=True, exist_ok=True)
          nome_arquivo = f"{str(y)} {c} {e} {d}.pdf"

          # Cria o caminho completo
          caminho_completo = pasta_destino / nome_arquivo  # O "/" junta os caminhos


          r = requests.get(url, headers=HEADERS, timeout=10)
          if r.status_code == 200:
              with open(caminho_completo, "wb") as f:
                  f.write(r.content)
              print(f"[SAVED] {caminho_completo}")
          else:
              print(f"[NO PDF] {url}")

for y2 in YEARS2:
  for c2 in CIRCUITS:
    for e2 in ETAPAS:
      for d2 in DOC:

          url2 = f"https://resources.motogp.com/files/results/{y2}/{c2}/MotoGP/{e2}/{d2}.pdf"

          pasta_destino = Path(f"motogp/{str(y2)}/{c2}")
          pasta_destino.mkdir(parents=True, exist_ok=True)
          nome_arquivo = f"{str(y2)} {c2} {e2} {d2}.pdf"
          caminho_completo = pasta_destino / nome_arquivo

          r2 = requests.get(url2, headers=HEADERS, timeout=10)
          if r2.status_code == 200:
              with open(caminho_completo, "wb") as f:
                  f.write(r2.content)
              print(f"[SAVED] {caminho_completo}")
          else:
              print(f"[NO PDF] {url}")

# Cria um objeto Path para uma pasta
pastaPrincipal = Path("/content/motogp")

anos = [2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010,
        2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019,
        2020, 2021, 2022, 2023, 2024, 2025, 2026]


for corrida in CIRCUITS:
  for s in anos:
    pastaAVerificar = pastaPrincipal/str(s)/str(corrida)

    try:
      os.rmdir(pastaAVerificar)
    except:
      continue


ROOT = Path("/content/motogp")

def extrair_dados(pdf_path):
    classification = []
    extras_not_classified = []
    extras_not_finished = []
    stats = {}
    ano = pdf_path.parts[-3]
    circuito = pdf_path.parts[-2]

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            texto = page.extract_text()
            if texto:
                for linha in texto.split("\n"):
                    # Classificação principal (Pos, Points, Number)
                    padrao = re.match(
                        r"^(\d+)\s+(\d*)\s+(\d+)\s+([A-Za-zÀ-ÿ'\- ]+)\s+([A-Z]{3})\s+(.+?)\s+([A-Za-zÀ-ÿ'\- ]+)\s+([\d'’:.]+)\s+([\d.]+|\d+\s+laps|Not finished.*)\s*(.*)$",
                        linha
                    )
                    if padrao:
                        pos, points, number, rider, nation, team, moto, total_time, kmh, gap = padrao.groups()
                        if points == "":
                            points = ""  # valor vazio para quem não tem pontuação
                        classification.append({
                            "Year": ano,
                            "Circuit": circuito,
                            "Pos": pos.strip(),
                            "Points": points.strip(),
                            "Number": number.strip(),
                            "Rider": rider.strip(),
                            "Nation": nation.strip(),
                            "Team": team.strip(),
                            "Motorcycle": moto.strip(),
                            "Total Time": total_time.strip(),
                            "Km/h": kmh.strip(),
                            "Gap": gap.strip()
                        })
                        continue

                    # Extras (Not Classified)
                    padrao_nc = re.match(
                        r"^(\d+)\s+([A-Za-zÀ-ÿ'\- ]+)\s+([A-Z]{3})\s+(.+?)\s+([A-Za-zÀ-ÿ'\- ]+)\s+([\d'’:.]+)\s+([\d.]+)\s+(.*)$",
                        linha
                    )
                    if padrao_nc:
                        number, rider, nation, team, moto, total_time, kmh, gap = padrao_nc.groups()
                        extras_not_classified.append({
                            "Number": number.strip(),
                            "Rider": rider.strip(),
                            "Nation": nation.strip(),
                            "Team": team.strip(),
                            "Motorcycle": moto.strip(),
                            "Total Time": total_time.strip(),
                            "Km/h": kmh.strip(),
                            "Gap": gap.strip()
                        })
                        continue

                    # Extras (Not finished first lap)
                    padrao_nf = re.match(
                        r"^(\d+)\s+([A-Za-zÀ-ÿ'\- ]+)\s+([A-Z]{3})\s+(.+?)\s+([A-Za-zÀ-ÿ'\- ]+)$",
                        linha
                    )
                    if padrao_nf:
                        number, rider, nation, team, moto = padrao_nf.groups()
                        extras_not_finished.append({
                            "Number": number.strip(),
                            "Rider": rider.strip(),
                            "Nation": nation.strip(),
                            "Team": team.strip(),
                            "Motorcycle": moto.strip()
                        })
                        continue

                    # Estatísticas extras
                    if "Pole Position" in linha:
                        stats["Pole Position"] = linha
                    elif "Fastest Lap" in linha:
                        stats["Fastest Lap"] = linha
                    elif "Circuit Record Lap" in linha:
                        stats["Circuit Record Lap"] = linha
                    elif "Circuit Best Lap" in linha:
                        stats["Circuit Best Lap"] = linha

    # Estrutura final
    corrida = {
        "Year": ano,
        "Circuit": circuito,
        "Classification": classification,
        "Extras": {
            "Not Classified": extras_not_classified,
            "Not Finished First Lap": extras_not_finished
        },
        "Stats": stats
    }

    # salvar JSON único
    json_path = pdf_path.with_suffix(".full.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(corrida, f, indent=4, ensure_ascii=False)
    print(f"[FULL JSON SAVED] {json_path}")

    # salvar CSV da classificação
    if classification:
        df = pd.DataFrame(classification)
        csv_path = pdf_path.with_suffix(".full.csv")
        df.to_csv(csv_path, index=False)
        print(f"[FULL CSV SAVED] {csv_path}")

    return classification

# Consolidado
all_data = []
for pdf_file in ROOT.rglob("*.pdf"):
    classificacao = extrair_dados(pdf_file)
    if classificacao:
        all_data.extend(classificacao)

# salvar mestre CSV/JSON
if all_data:
    master_df = pd.DataFrame(all_data)
    master_df.to_csv("motogp_master_full.csv", index=False)
    with open("motogp_master_full.json", "w", encoding="utf-8") as f:
        json.dump(all_data, f, indent=4, ensure_ascii=False)
    print("[MASTER CREATED] motogp_master_full.csv, motogp_master_full.json")

# Compactar tudo
final_zip = Path("motogp_total.zip")
with zipfile.ZipFile(final_zip, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in ROOT.rglob("*"):
        if file_path.is_file():
            arcname = file_path.relative_to(ROOT.parent)
            zipf.write(file_path, arcname)
    zipf.write("motogp_master_full.csv")
    zipf.write("motogp_master_full.json")

print(f"[FINAL ZIP CREATED] {final_zip}")

[FULL JSON SAVED] /content/motogp/2009/VAL/2009 VAL RAC Classification.full.json
[CSV SAVED] /content/motogp/2009/VAL/2009 VAL RAC Classification.csv
[FULL JSON SAVED] /content/motogp/2009/VAL/2009 VAL QP Classification.full.json
[FULL JSON SAVED] /content/motogp/2009/JPN/2009 JPN RAC Classification.full.json
[CSV SAVED] /content/motogp/2009/JPN/2009 JPN RAC Classification.csv
[FULL JSON SAVED] /content/motogp/2009/JPN/2009 JPN QP Classification.full.json
[FULL JSON SAVED] /content/motogp/2009/ITA/2009 ITA QP Classification.full.json
[FULL JSON SAVED] /content/motogp/2009/ITA/2009 ITA RAC Classification.full.json
[CSV SAVED] /content/motogp/2009/ITA/2009 ITA RAC Classification.csv
[FULL JSON SAVED] /content/motogp/2009/USA/2009 USA RAC Classification.full.json
[CSV SAVED] /content/motogp/2009/USA/2009 USA RAC Classification.csv
[FULL JSON SAVED] /content/motogp/2009/USA/2009 USA QP Classification.full.json
[FULL JSON SAVED] /content/motogp/2009/QAT/2009 QAT RAC Classification.full.json